In [29]:
import json
import tensorflow as tf
import h5py
from keras.layers import TextVectorization
from keras.models import load_model


# =========================
# CONFIG
# =========================

with open("../config/config.json", "r") as f:
    config = json.load(f)

MAX_LEN = config["max_len"]


# =========================
# CAPTIONS
# =========================

with open("../data/captions/captions_split_clean.json", "r") as f:
    captions = json.load(f)


train_captions = [
    cap
    for caps_list in captions["train"].values()
    for cap in caps_list
]


def truncate_captions(captions, max_len):
    out = []
    for c in captions:
        words = c.split()
        out.append(" ".join(words[:max_len]) if len(words) > max_len else c)
    return out


train_captions = truncate_captions(train_captions, MAX_LEN - 2)
train_captions = [f"[START] {c} [END]" for c in train_captions]


# =========================
# VECTORIZER
# =========================

config["vec_config"]["output_sequence_length"] = MAX_LEN

vectorizer = TextVectorization(**config["vec_config"])
vectorizer.adapt(train_captions)


vocab = vectorizer.get_vocabulary()
id_to_word = {i: w for i, w in enumerate(vocab)}
word_to_id = {w: i for i, w in enumerate(vocab)}


# =========================
# MODEL
# =========================

icg = load_model("../output/best_model.keras")


# =========================
# IMAGE FEATURES
# =========================

with open("../data/images/test_images.txt", "r") as f:
    test_images = [line.strip() for line in f][:5]


with h5py.File("../data/images/flickr30k_vgg16_features.h5", "r") as f:
    test_features = tf.stack([
        f[img_id][()] for img_id in test_images # type: ignore
    ])


# =========================
# GROUND TRUTH TEST CAPTIONS
# =========================

test_ground_truth = {
    img_id: captions["test"][img_id][0]
    for img_id in test_images
}


# =========================
# GENERATION FUNCTION
# =========================

def generate_caption(model, img_features, vectorizer, max_len=MAX_LEN):

    vocab = vectorizer.get_vocabulary()
    word_to_id = {w: i for i, w in enumerate(vocab)}
    id_to_word = {i: w for i, w in enumerate(vocab)}

    start_id = word_to_id["[START]"]
    end_id = word_to_id["[END]"]

    sequence = [start_id]

    img_features = tf.expand_dims(img_features, axis=0)

    for _ in range(max_len):

        padded = tf.keras.preprocessing.sequence.pad_sequences( # type: ignore
            [sequence],
            maxlen=max_len,
            padding="post"
        )

        pred = model.predict((img_features, padded), verbose=0)

        last_real = len(sequence) - 1
        probs = pred[0, last_real]

        next_id = tf.argmax(probs).numpy()

        if next_id == end_id:
            break

        sequence.append(next_id)

    caption = " ".join(
        id_to_word[i]
        for i in sequence
        if i not in (start_id, end_id)
    )

    return caption


# =========================
# INFERENCE
# =========================

for img_id, img_feat in zip(test_images, test_features):

    caption = generate_caption(icg, img_feat, vectorizer)

    print(f"\nImage ID: {img_id}")
    print(f"Ground Truth: {test_ground_truth[img_id]}")
    print(f"Generated: {caption}")


Image ID: 4963789758
Ground Truth: older woman featured in fore ground of large race that number of people running in
Generated: two women in white shirts and black pants walking down street

Image ID: 4627601777
Ground Truth: man and young boy wearing striped shirts walk on dirt path along water
Generated: man in blue shirt and guitar walking down street

Image ID: 2787372148
Ground Truth: four guys gathering around guy wearing navy blue that fixing something
Generated: man in blue shirt and blue guitar sitting on bench

Image ID: 3589052481
Ground Truth: man looking at camera posing for photo with arm around woman who looking away
Generated: man in blue shirt and woman in black shirt and black pants standing in front of riding

Image ID: 439823342
Ground Truth: woman selling fresh vegetables on streets in cart
Generated: man in blue shirt and guitar sitting on bench
